# Data Generation for Synthetic RCT Diagnostics

This project now treats synthetic data as a reusable on-disk artifact rather than an in-memory side effect.

The default workflow is:

1. Check whether `data/synthetic_experiments.jsonl` already exists.
2. If it exists, load it.
3. If it does not exist, generate it once and save it.

That means notebooks, scripts, and evaluation runs all read from the same dataset by default.

In [ ]:
from rct_diagnosis_agent.data import DEFAULT_DATASET_PATH, ensure_dataset, load_experiments

dataset_path = ensure_dataset(dataset_path=DEFAULT_DATASET_PATH, count=25, seed=7)
experiments = load_experiments(dataset_path)
dataset_path, len(experiments)

## What is stored in the dataset

Each line in the JSONL file is one `ExperimentSummary` object containing:

- group sizes
- pre-period means and standard deviations
- post-period means and standard deviations
- conversion rates
- optional segment summaries
- hidden ground-truth issue labels used for evaluation

The file is compact, inspectable, and reusable across the whole project.

In [ ]:
experiments[0].model_dump()

## Failure modes represented in the stored dataset

The generator can inject several common experiment issues:

- `srm`
- `pre_period_imbalance`
- `low_statistical_power`
- `outliers`
- `simpsons_paradox`

Because the dataset is saved to disk, the exact same synthetic experiments can be reused later for agent walkthroughs and evaluation.

In [ ]:
from collections import Counter

issue_counts = Counter(issue for exp in experiments for issue in exp.hidden_truth)
issue_counts

## Visualizing one outlier-heavy experiment

Outlier risk is encoded as a large jump in post-period variability. We can inspect one matching experiment directly from the stored dataset.

In [ ]:
import matplotlib.pyplot as plt

outlier_example = next(exp for exp in experiments if "outliers" in exp.hidden_truth)
labels = ["control_pre", "control_post", "treatment_pre", "treatment_post"]
values = [
    outlier_example.control_pre_std,
    outlier_example.control_post_std,
    outlier_example.treatment_pre_std,
    outlier_example.treatment_post_std,
]

plt.figure(figsize=(8, 4))
plt.bar(labels, values, color=["#4C78A8", "#F58518", "#54A24B", "#E45756"])
plt.title("Variance profile for a stored outlier-heavy synthetic experiment")
plt.ylabel("Standard deviation")
plt.xticks(rotation=15)
plt.show()